# AtendeAI — Fase 2: do protótipo ao agente (ReAct, ferramentas e prompts avançados)
Continua a Fase 1. **Uso de IA:** ver `docs/uso_de_ia.md`.

**Evidências desta fase (checklist do enunciado):**
- [ ] Comparação de orquestradores e justificativa do LangChain (seção 1)
- [ ] Ferramentas, incluindo a **API externa ViaCEP**, e demonstração de function calling (seção 2)
- [ ] Prompt PTCF com few-shot e chain-of-thought (seção 3)
- [ ] Agente ReAct (`initialize_agent` + `ZERO_SHOT_REACT_DESCRIPTION`) com memória e callbacks (seção 4)
- [ ] Chains sequenciais e paralelas (seção 5)
- [ ] Roteamento dinâmico com `RouterChain` (seção 6)
- [ ] Parser com tratamento de erro, memória, FAISS e callbacks, com justificativas (seção 7)

> **Importante:** no modo `dummy` as decisões do agente vêm de regras fixas (serve para reproduzir o projeto sem credenciais). A demonstração de raciocínio de verdade, e o vídeo, exigem `LLM_MODE=openai`.

## 1. Escolha do orquestrador

*Avaliação qualitativa do autor com base na documentação pública de cada projeto. Confirme os pontos na documentação oficial e cite as fontes no relatório.*

| Critério | LangChain | LlamaIndex | CrewAI | AutoGen |
|---|---|---|---|---|
| **Foco principal** | Blocos genéricos para apps com LLM (prompts, parsers, ferramentas, chains, agentes) | Dados e RAG: indexação e consulta | Equipes de agentes com papéis e tarefas | Conversas entre múltiplos agentes |
| **Autonomia** | Configurável: loop do agente com ferramentas, com limite de iterações | Agentes disponíveis, mas o centro é o acesso a dados | Alta, distribuída entre agentes | Alta, via conversa entre agentes |
| **Abstração** | Média: o desenvolvedor vê e controla cada etapa | Alta para RAG | Alta: papéis e tarefas escondem o loop | Alta: o protocolo de conversa guia o fluxo |
| **Modularidade** | Alta: componentes pequenos e intercambiáveis (Runnables) | Alta na camada de dados | Média: peças pensadas para o modelo de equipe | Média |
| **Ajuste ao AtendeAI hoje** | Agente **único** com SQL, RAG e API, com controle fino | Bom só para a parte de RAG | Excesso de estrutura para 1 agente | Idem |

**Decisão: LangChain.** (1) O AtendeAI é, nesta etapa, **um agente único** que precisa de controle fino sobre prompt, parser, ferramentas e memória, e esse é o nível de abstração do LangChain. (2) O enunciado exige as construções do próprio LangChain (`initialize_agent`, `AgentExecutor`, `RunnableWithMessageHistory`) e a avaliação com LangSmith, que se integra nativamente. (3) A composição por Runnables (decidida na Fase 1) mantém o agente reutilizável como **um nó** quando a próxima disciplina montar uma arquitetura com vários agentes. **Trade-off aceito:** as APIs `initialize_agent`, `RouterChain` e `ConversationBufferMemory` exigidas são legadas e geram avisos de depuração; por isso as versões estão limitadas a `0.3.x`.

In [ ]:
import os, sys, time, warnings
from pathlib import Path
warnings.filterwarnings("ignore", category=DeprecationWarning)   # APIs legadas exigidas pelo enunciado

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv
load_dotenv(ROOT / ".env")

from src.llm_factory import get_llm
from src.chains import build_chain_intencao
from src.tools import FERRAMENTAS, consultar_cep, calcular_prazo_frete, buscar_politica
from src.callbacks import LogHandler
from src.indice import construir_indice, indice_dir

llm = get_llm()
print("Modo:", os.getenv("LLM_MODE", "dummy"), "|", type(llm).__name__)
print("Índice de políticas:", construir_indice())   # só prepara se não existir (tratado como índice pré-existente)

## 2. Ferramentas e function calling

| Ferramenta | Tipo | O que faz | Entrada |
|---|---|---|---|
| `consultar_cep` | **API externa** (ViaCEP) | Valida o CEP e devolve o endereço | `01310-100` |
| `calcular_prazo_frete` | **Hardcoded** (regra fixa, tabela da loja) | Prazo e frete por região do CEP | `CEP` ou `CEP;valor` |
| `buscar_politica` | **Semântica** (RAG sobre FAISS) | Trechos das políticas relevantes à pergunta | pergunta em texto |

Todas recebem **uma única string**, exigência do `ZERO_SHOT_REACT_DESCRIPTION`. SQL (text-to-query) entra na Fase 4.

In [ ]:
print(consultar_cep.invoke("01310-100"))
print(calcular_prazo_frete.invoke("01310-100;150.00"))
print(calcular_prazo_frete.invoke("80010-000;250"))
print(buscar_politica.invoke("Posso devolver um produto depois de receber?"))

### 2.1 Function calling
O modelo recebe o **esquema** das ferramentas (`bind_tools`) e, em vez de texto, devolve uma *chamada estruturada*; quem executa a ferramenta é o nosso código. É o mecanismo que o agente da Fase 5 (`create_openai_tools_agent`) usa. O agente ReAct desta fase usa o mesmo conjunto de ferramentas, mas decide por texto (`Thought/Action`).

In [ ]:
llm_tools = llm.bind_tools([consultar_cep, calcular_prazo_frete])
resp = llm_tools.invoke("Qual é o endereço do CEP 01310-100?")
print("tool_calls:", resp.tool_calls)

for tc in resp.tool_calls:                                  # nosso código executa a chamada pedida pelo modelo
    print("resultado:", {"consultar_cep": consultar_cep, "calcular_prazo_frete": calcular_prazo_frete}[tc["name"]].invoke(tc["args"]))

## 3. Prompt de sistema com PTCF, few-shot e chain-of-thought
O prompt fica em `src/prompts_react.py`. Estrutura:
- **Persona**: atendente Lia, tom cordial, nunca inventa dados.
- **Task**: resolver o pedido, decompor casos compostos, admitir quando falta ferramenta.
- **Context**: o agente só sabe o que as ferramentas retornam; usar o histórico para referências como "esse CEP".
- **Format**: um *Thought* curto antes de cada ação (**chain-of-thought explícito e depurável**), resposta final em português, citando a fonte, e pedir dados que faltam.
- **Few-shot**: dois exemplos completos de `Thought/Action/Observation`, um com duas ferramentas encadeadas e outro de política.

In [ ]:
from src.prompts_react import PREFIX, SUFFIX
print(PREFIX)
print("...")
print(SUFFIX[:600], "...")

## 4. Agente ReAct com memória e callbacks

In [ ]:
from src.agente import criar_agente, criar_memoria

handler = LogHandler()
memoria = criar_memoria(k=6)
agente = criar_agente(llm, FERRAMENTAS, memoria=memoria, verbose=True)
CFG = {"callbacks": [handler]}   # callbacks no invoke sao herdados pelas ferramentas e pelo LLM

r1 = agente.invoke({"input": "Qual o prazo e o frete para o CEP 01310-100?"}, config=CFG)
print("\nRESPOSTA:", r1["output"])

In [ ]:
# 2º turno: "esse CEP" só faz sentido com o histórico da conversa
r2 = agente.invoke({"input": "E qual é o endereço desse CEP?"}, config=CFG)
print("\nRESPOSTA:", r2["output"])
print("\nFerramentas usadas até aqui:", handler.ferramentas_usadas())

In [ ]:
r3 = agente.invoke({"input": "Posso devolver um produto depois de receber?"}, config=CFG)
print("\nRESPOSTA:", r3["output"])

## 5. Chains paralelas e sequenciais (LCEL)
Pedido composto: *"Comprei um tênis, veio com defeito há 20 dias. Quero trocar e saber o prazo para o CEP 01310-100."* Três análises **independentes** rodam em **paralelo** (intenção, validação do CEP, busca de política) e depois um passo **sequencial** monta o dossiê e decide o próximo passo.

In [ ]:
import re
from langchain_core.runnables import RunnableParallel, RunnableLambda
from src.heuristicas import _CEP

chain_intencao = build_chain_intencao(llm)

def _cep(x):
    m = _CEP.search(x["mensagem"]); return f"{m.group(1)}-{m.group(2)}" if m else None

paralelo = RunnableParallel(
    analise=chain_intencao,
    endereco=RunnableLambda(lambda x: consultar_cep.invoke(_cep(x)) if _cep(x) else "CEP não informado"),
    politica=RunnableLambda(lambda x: buscar_politica.invoke(x["mensagem"])),
)

PROXIMO = {"prazo_frete": "agente_frete", "troca_devolucao": "agente_politica", "cancelamento": "agente_politica"}
def montar_dossie(d):
    return {**d, "proximo_passo": PROXIMO.get(d["analise"].intencao, "atendente_humano")}

pipeline = paralelo | RunnableLambda(montar_dossie)      # paralelo -> sequencial

msg = "Comprei um tênis, veio com defeito há 20 dias. Quero trocar e saber o prazo para o CEP 01310-100."
t0 = time.perf_counter()
dossie = pipeline.invoke({"mensagem": msg})
print(f"pipeline em {time.perf_counter() - t0:.2f}s")
print("intenção:", dossie["analise"].intencao, "| entidades:", dossie["analise"].entidades.model_dump())
print("endereço:", dossie["endereco"])
print("política:", dossie["politica"][:200], "...")
print("próximo passo:", dossie["proximo_passo"])

## 6. Roteamento dinâmico com `RouterChain`
O roteador reutiliza a **chain de intenção do MVP** para decidir, em tempo de execução, qual chain atende a mensagem. O destino padrão é uma chain sem LLM que encaminha ao humano (o status de pedido só ganha ferramenta na Fase 4).

In [ ]:
from typing import Any, Dict, Optional
from langchain.chains.router.base import MultiRouteChain, RouterChain
from langchain.chains.transform import TransformChain

class RoteadorPorIntencao(RouterChain):
    classificador: Any
    mapa: Dict[str, str]

    @property
    def input_keys(self):
        return ["input"]

    def _call(self, inputs: Dict[str, Any], run_manager=None) -> Dict[str, Any]:
        analise = self.classificador.invoke({"mensagem": inputs["input"]})
        return {"destination": self.mapa.get(analise.intencao), "next_inputs": {"input": inputs["input"]}}

# destinos: dois agentes com subconjuntos diferentes de ferramentas (memória compartilhada)
agente_frete = criar_agente(llm, [consultar_cep, calcular_prazo_frete], memoria=memoria, verbose=False)
agente_politica = criar_agente(llm, [buscar_politica], memoria=memoria, verbose=False)

padrao = TransformChain(
    input_variables=["input"], output_variables=["output"],
    transform=lambda d: {"output": "Ainda não tenho ferramenta para isso. Vou encaminhar você a um atendente humano."},
)

roteador = MultiRouteChain(
    router_chain=RoteadorPorIntencao(classificador=chain_intencao,
                                     mapa={"prazo_frete": "prazo_frete", "troca_devolucao": "politica", "cancelamento": "politica"}),
    destination_chains={"prazo_frete": agente_frete, "politica": agente_politica},
    default_chain=padrao,
)

testes = [
    "Qual o prazo para o CEP 80010-000?",
    "Quero cancelar meu pedido, ainda dá?",
    "Onde está o meu pedido 10234?",       # status_pedido -> sem ferramenta ainda -> padrão
]
for t in testes:
    out = roteador.invoke({"input": t}, config=CFG)
    print(f"> {t}\n  {out['output'][:220]}\n")

## 7. Confiabilidade: parser com erro, memória, FAISS e callbacks

### 7.1 Parser estruturado com tratamento de erros
A chain de intenção (`src/chains.py`) usa `parse_seguro`: tenta o parser; se falhar, **extrai o JSON do texto** (cercas ```` ```json ````, texto extra); se ainda falhar, devolve um **fallback seguro** (intenção `outro`, confiança 0), sem derrubar o agente.

In [ ]:
from langchain_core.output_parsers import PydanticOutputParser
from src.parsing import parse_seguro
from src.schemas import AtendimentoOutput

parser = PydanticOutputParser(pydantic_object=AtendimentoOutput)
casos = {
  "JSON limpo": '{"intencao":"prazo_frete","entidades":{},"requer_ferramenta":true,"resposta":"ok","confianca":0.8}',
  "com cerca e texto extra": 'Claro! ```json\n{"intencao":"cancelamento","entidades":{"numero_pedido":"10250"},"requer_ferramenta":true,"resposta":"ok","confianca":0.7}\n``` Espero ajudar.',
  "intenção inválida": '{"intencao":"xyz","entidades":{},"requer_ferramenta":false,"resposta":"ok","confianca":0.5}',
  "texto livre": "Não sei o que responder.",
}
for nome, txt in casos.items():
    r = parse_seguro(txt, parser)
    print(f"{nome:26s} -> intencao={r.intencao:14s} confianca={r.confianca}")

### 7.2 Callbacks: trilha de execução do agente

In [ ]:
import pandas as pd
colunas = ["tipo", "ferramenta", "entrada", "duracao_s", "saida", "resposta"]
eventos = [e for e in handler.eventos if e["tipo"] in ("agent_action", "tool_end", "agent_finish")]
pd.DataFrame(eventos).reindex(columns=colunas).fillna("")

### 7.3 Memória atual do agente

In [ ]:
print(memoria.load_memory_variables({})["chat_history"][-700:])

### 7.4 Justificativas das escolhas de confiabilidade

| Componente | Escolha | Por quê | Limitação |
|---|---|---|---|
| Parser | `PydanticOutputParser` + `parse_seguro` (extrai JSON, fallback seguro) | A saída vira objeto validado; saídas malformadas são frequentes em LLMs e não podem derrubar o atendimento | O fallback perde informação; com LLM real, avaliar também `OutputFixingParser` |
| Memória | `ConversationBufferWindowMemory(k=6)` | Atendimento tem sessões curtas; a janela limita tokens e custo e não exige chamar o LLM para resumir | Esquece o que sai da janela; a Fase 3 trata o longo prazo e `ConversationBufferMemory` |
| FAISS como ferramenta | `buscar_politica` sobre as 6 políticas | Prazos e regras mudam e não devem vir da memória do modelo; RAG dá fonte verificável | O índice é pequeno e a qualidade depende dos embeddings; falhas de recuperação são diagnosticadas na Fase 4 |
| Callbacks | `LogHandler` (ações, ferramentas, duração, erros), passado em `invoke(config=...)` | Mostra *o que o agente decidiu* e quanto cada ferramenta custou, base para depurar e para a trajetória no LangSmith. Passar no `invoke` (e não no construtor) faz as ferramentas e o LLM herdarem o callback | Logs locais; o LangSmith entra na Fase 5 |
| `handle_parsing_errors` e `max_iterations=6` | Ativados no executor | Evita laço infinito e permite ao agente se recuperar de um formato inválido | Pode encerrar cedo em tarefas muito longas |

**No modo dummy**, os embeddings são lexicais (hashing), não semânticos, e as decisões do agente vêm de regras. Use `LLM_MODE=openai` para as evidências do relatório.

### 7.5 Experimento A/B do prompt (medido com o LLM real)
Hipótese inicial: uma regra explícita de formato ("uma ação por vez, pare depois de `Action Input`") reduziria erros de formato. O script `scripts/diagnostico_react.py` comparou o prompt **com** e **sem** a regra, com as mesmas duas perguntas e o mesmo modelo (`gemini-3.5-flash-lite`):

| Variante | Chamadas ao LLM | Erros de formato | Respostas vazias | Resultado |
|---|---|---|---|---|
| Com a regra | 9 | 6 | 6 | 1 de 2 perguntas estourou o limite de iterações |
| **Sem a regra** | 5 | 0 | 0 | 2 de 2 respostas corretas |

**Conclusão:** a regra *piorou* o comportamento (o modelo passou a devolver respostas vazias) e foi **removida** do prompt. Amostra pequena (2 perguntas por variante), mas a diferença é grande e consistente com as execuções anteriores. A causa exata não foi confirmada. Lição para o relatório: mudanças de prompt devem ser validadas com medição, não assumidas como melhorias.

## 8. Resumo e próximos passos
- Agente ReAct funcional com 3 ferramentas (uma API externa real), memória e callbacks.
- Prompt PTCF com few-shot e chain-of-thought.
- Pipeline paralelo + sequencial e `RouterChain` com agentes de ferramentas diferentes.
- **Para o relatório:** rode também com `LLM_MODE=openai` e guarde os logs `verbose` das seções 4 e 6 (o raciocínio `Thought/Action` é a evidência principal).
- **Fase 3:** `ConversationBufferMemory`, memória de longo prazo (CoALA), intenção e entidades em conversa multi-turno.